# 🚀 Bank_Churn_Data_Preprocessing – Applying Days 1–3 Skills

**Goal:** Apply Days 1–3 skills to real bank customer churn data

**Dataset:** `Bank_Churn.csv` (CustomerId, Surname, CreditScore, Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary, Exited)

---

## 🧠 Quick Recap

We learned:
- **Day 1:** Comprehensions & Generators
- **Day 2:** `map`/`filter`/`reduce`/`partial`
- **Day 3:** Decorators & Closures

**Today we use all of them on real data!**

---

In [1]:
# 📊 Load the Bank Churn Dataset
import pandas as pd  # Import pandas for data manipulation

# Read the CSV file into a DataFrame
# Make sure Bank_Churn.csv is in the same directory as this notebook
df = pd.read_csv('Bank_Churn.csv')

# Display the first 5 rows to understand the data structure
print("=== First 5 Rows ===")
print(df.head())

# Display information about the DataFrame (column types, non-null counts)
print("\n=== DataFrame Info ===")
print(df.info())

# Display the shape of the DataFrame (rows, columns)
print("\n=== DataFrame Shape ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

=== First 5 Rows ===
   CustomerId   Surname  CreditScore Geography  Gender  Age  Tenure  \
0    15634602  Hargrave          619    France  Female   42       2   
1    15647311      Hill          608     Spain  Female   41       1   
2    15619304      Onio          502    France  Female   42       8   
3    15701354      Boni          699    France  Female   39       1   
4    15737888  Mitchell          850     Spain  Female   43       2   

     Balance  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  \
0       0.00              1          1               1        101348.88   
1   83807.86              1          0               1        112542.58   
2  159660.80              3          1               0        113931.57   
3       0.00              2          0               0         93826.63   
4  125510.82              1          1               1         79084.10   

   Exited  
0       1  
1       0  
2       1  
3       0  
4       0  

=== DataFrame Info ===
<clas

## ⚡ Day 1 Skills – Comprehensions & Generators

Let's practice list comprehensions, generators, and dict comprehensions on our bank data!

---

In [2]:
# 🔧 Day 1: Comprehensions & Generators in Action

# 1️⃣ List Comprehension → Find high-risk customers (Age > 50 AND Balance == 0)
# High-risk customers are older people with no balance - potential churn risk!
high_risk_customers = [
    row['CustomerId']  # We only want the CustomerId
    for _, row in df.iterrows()  # Iterate through each row
    if row['Age'] > 50 and row['Balance'] == 0  # Filter condition
]
print(f"🚨 High-risk customers (Age > 50 & Balance = 0): {len(high_risk_customers)} found")
print(f"First 10 IDs: {high_risk_customers[:10]}")

# 2️⃣ Generator → Lazy iteration over customers with NumOfProducts >= 3
# Generators are memory-efficient - they don't load everything at once!
high_product_customers = (
    row['CustomerId']  # Yield CustomerId
    for _, row in df.iterrows()  # Iterate through rows
    if row['NumOfProducts'] >= 3  # Filter condition
)

# Convert generator to list to see results (in real use, we'd iterate lazily)
high_product_list = list(high_product_customers)
print(f"\n📦 Customers with 3+ products: {len(high_product_list)} found")

# 3️⃣ Dict Comprehension → {CustomerId: Age} for first 100 customers
# Dictionary comprehensions are great for quick lookups!
customer_age_dict = {
    row['CustomerId']: row['Age']  # Key: CustomerId, Value: Age
    for _, row in df.head(100).iterrows()  # Only first 100 rows
}
print(f"\n📋 Customer-Age dictionary (first 5): {dict(list(customer_age_dict.items())[:5])}")

# 4️⃣ Generator Expression → Count customers with Balance > 100000 WITHOUT loading full list
# This is memory efficient - we only keep a counter, not the whole list!
wealthy_count = sum(1 for _, row in df.iterrows() if row['Balance'] > 100000)
print(f"\n💰 Customers with Balance > $100,000: {wealthy_count}")
print("✅ Used generator expression - no full list stored in memory!")

🚨 High-risk customers (Age > 50 & Balance = 0): 414 found
First 10 IDs: [15623944, 15614049, 15805254, 15804919, 15671137, 15713483, 15679145, 15789669, 15656176, 15774854]

📦 Customers with 3+ products: 326 found

📋 Customer-Age dictionary (first 5): {15634602: 42, 15647311: 41, 15619304: 42, 15701354: 39, 15737888: 43}

💰 Customers with Balance > $100,000: 4799
✅ Used generator expression - no full list stored in memory!


## 🔄 Day 2 Skills – `map`, `filter`, `reduce`, `partial`

Functional programming tools make data transformations clean and reusable!

---

In [ ]:
# 🔧 Day 2: Functional Programming with map, filter, reduce, partial
from functools import reduce, partial  # Import reduce and partial from functools

# 1️⃣ map + lambda → Create normalized EstimatedSalary (divide by max salary)
# Normalization scales values between 0 and 1 - common in ML preprocessing!
max_salary = df['EstimatedSalary'].max()  # Find the maximum salary
print(f"💵 Maximum Estimated Salary: ${max_salary:,.2f}")

# Use map with lambda to normalize each salary value
# map applies the function to every element in the iterable
normalized_salaries = list(map(
    lambda salary: salary / max_salary,  # Normalize by dividing by max
    df['EstimatedSalary']  # Input iterable
))
print(f"📊 First 5 normalized salaries: {normalized_salaries[:5]}")

# 2️⃣ filter → Keep only active members (IsActiveMember == 1)
# filter keeps only elements where the function returns True
active_members = list(filter(
    lambda row: row['IsActiveMember'] == 1,  # Keep only active members
    [row for _, row in df.iterrows()]  # Convert DataFrame rows to list of dicts
))
print(f"\n✅ Active members: {len(active_members)} out of {len(df)}")

# 3️⃣ partial → Create reusable function for high-balance filtering
# partial lets us 'freeze' some arguments - great for reusable filters!
def has_high_balance(customer, threshold=0):
    """Check if customer has balance above threshold."""
    return customer['Balance'] > threshold

# Create a specialized function with threshold frozen at 100,000
high_balance_filter = partial(has_high_balance, threshold=100000)

# Use the partial function with filter
high_balance_customers = list(filter(
    high_balance_filter,  # Our partial function
    [row for _, row in df.iterrows()]  # Input data
))
print(f"\n🏦 High balance customers (>$100k): {len(high_balance_customers)}")

# 4️⃣ Combine filter → map → reduce for age group analysis
# This pipeline: 1) Filter invalid ages, 2) Map to categories, 3) Reduce to count

# Step 1: Filter out invalid ages (< 18 or > 100)
valid_age_customers = list(filter(
    lambda row: 18 <= row['Age'] <= 100,
    [row for _, row in df.iterrows()]
))
print(f"\n👥 Valid age customers: {len(valid_age_customers)}")

# Step 2: Map ages to categories
def categorize_age(row):
    """Categorize age into Young, Adult, or Senior."""
    age = row['Age']
    if age < 30:
        return 'Young'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

age_categories = list(map(categorize_age, valid_age_customers))
print(f"📊 Age categories sample: {age_categories[:10]}")

# Step 3: Reduce to count totals in each category
# reduce applies a function cumulatively to reduce iterable to single value
def count_categories(counts, category):
    """Count occurrences of each category."""
    counts[category] = counts.get(category, 0) + 1  # Increment count
    return counts

category_counts = reduce(count_categories, age_categories, {})  # Start with empty dict
print(f"\n📈 Age Group Counts: {category_counts}")

💵 Maximum Estimated Salary: $199,992.48
📊 First 5 normalized salaries: [np.float64(0.5067634543058819), np.float64(0.5627340588006109), np.float64(0.5696792699405497), np.float64(0.4691507900697066), np.float64(0.3954353683698507)]

✅ Active members: 5151 out of 10000

🏦 High balance customers (>$100k): 4799

👥 Valid age customers: 10000
📊 Age categories sample: ['Adult', 'Adult', 'Adult', 'Adult', 'Adult', 'Adult', 'Adult', 'Young', 'Adult', 'Young']

📈 Age Group Counts: {'Adult': 7833, 'Young': 1641, 'Senior': 526}


## 🎁 Day 3 Skills – Decorators & Closures

Decorators add functionality to functions. Closures create function factories. Both are essential for professional Python code!

---

In [4]:
# 🔧 Day 3: Decorators & Closures
import time  # Import time module for timing decorator

# 1️⃣ Timing Decorator - Measure how long functions take to execute
def timing_decorator(func):
    """Decorator that measures and prints execution time."""
    def wrapper(*args, **kwargs):  # *args, **kwargs allow any arguments
        start_time = time.time()  # Record start time
        result = func(*args, **kwargs)  # Execute the function
        end_time = time.time()  # Record end time
        elapsed = end_time - start_time  # Calculate elapsed time
        print(f"⏱️  {func.__name__} took {elapsed:.4f} seconds")
        return result  # Return the original result
    return wrapper  # Return the wrapper function

# 2️⃣ Logging Decorator - Track what the function is doing
def logging_decorator(func):
    """Decorator that logs start and end of function execution."""
    def wrapper(*args, **kwargs):
        print(f"🚀 Starting {func.__name__}...")
        result = func(*args, **kwargs)  # Execute the function
        # If result is a DataFrame, count rows; otherwise use len() if possible
        if hasattr(result, 'shape'):
            count = result.shape[0]  # For DataFrames
        elif hasattr(result, '__len__'):
            count = len(result)  # For lists, etc.
        else:
            count = 'N/A'
        print(f"✅ Finished {func.__name__} – processed {count} rows")
        return result
    return wrapper

# 3️⃣ Closure-based Function Factory - Create custom age filters
def create_age_filter(min_age):
    """
    Closure that returns a function to check if age >= min_age.
    The min_age is 'remembered' even after this function returns!
    """
    def age_checker(customer):
        """Check if customer's age meets the minimum."""
        return customer['Age'] >= min_age
    return age_checker  # Return the inner function (closure)

# Create specialized age filters using our closure
is_adult = create_age_filter(18)  # Filter for adults (18+)
is_senior = create_age_filter(65)  # Filter for seniors (65+)

# Test the closures
sample_customer = {'Age': 70, 'CustomerId': 12345}
print(f"🔍 Customer age 70 is adult? {is_adult(sample_customer)}")
print(f"🔍 Customer age 70 is senior? {is_senior(sample_customer)}")

# 4️⃣ Apply BOTH decorators to a data cleaning function
@timing_decorator  # This will measure time
@logging_decorator  # This will log start/end
def clean_data(dataframe):
    """
    Clean the data by:
    1. Removing rows with missing values
    2. Filtering out low credit scores (< 400)
    """
    # Make a copy to avoid modifying original
    cleaned = dataframe.copy()
    
    # Remove rows with any missing values
    cleaned = cleaned.dropna()
    
    # Filter out low credit scores using boolean indexing
    cleaned = cleaned[cleaned['CreditScore'] >= 400]
    
    return cleaned

# Run the decorated cleaning function
print("\n🧹 Running decorated clean_data function:")
cleaned_df = clean_data(df)

🔍 Customer age 70 is adult? True
🔍 Customer age 70 is senior? True

🧹 Running decorated clean_data function:
🚀 Starting clean_data...
✅ Finished clean_data – processed 9981 rows
⏱️  wrapper took 0.0167 seconds


## 🔗 Combined Pipeline – Putting It All Together

Now let's create a complete pipeline using ALL the concepts we've learned!

---

In [5]:
# 🔧 Combined Pipeline: All Skills Working Together

@timing_decorator
@logging_decorator
def comprehensive_analysis(dataframe):
    """
    Complete analysis pipeline using:
    - filter: Remove inactive members and low balance customers
    - map: Categorize ages
    - reduce: Count churned customers per category
    - generator: Process rows lazily
    """
    
    # Step 1: Use generator to iterate lazily over rows
    # Generators are memory-efficient for large datasets!
    row_generator = (row for _, row in dataframe.iterrows())
    
    # Step 2: Filter - Keep only active members with balance > 0
    # These are engaged customers who actually use the bank
    active_with_balance = filter(
        lambda row: row['IsActiveMember'] == 1 and row['Balance'] > 0,
        row_generator
    )
    
    # Step 3: Map - Categorize ages and add churn status
    def enrich_customer_data(row):
        """Add age category and keep relevant fields."""
        age = row['Age']
        # Determine age category
        if age < 30:
            category = 'Young'
        elif age < 60:
            category = 'Adult'
        else:
            category = 'Senior'
        
        # Return enriched data as dictionary
        return {
            'CustomerId': row['CustomerId'],
            'Age': age,
            'Category': category,
            'Exited': row['Exited'],  # 1 = churned, 0 = stayed
            'Balance': row['Balance']
        }
    
    enriched_customers = list(map(enrich_customer_data, active_with_balance))
    
    # Step 4: Reduce - Count churned customers per category
    def count_churned_per_category(acc, customer):
        """
        Accumulate counts of churned vs retained per category.
        acc is a dictionary: {category: {'churned': X, 'retained': Y}}
        """
        category = customer['Category']
        
        # Initialize category if not exists
        if category not in acc:
            acc[category] = {'churned': 0, 'retained': 0, 'total_balance': 0}
        
        # Increment appropriate counter
        if customer['Exited'] == 1:
            acc[category]['churned'] += 1
        else:
            acc[category]['retained'] += 1
        
        # Sum up balance for average calculation
        acc[category]['total_balance'] += customer['Balance']
        
        return acc
    
    results = reduce(count_churned_per_category, enriched_customers, {})
    
    # Calculate churn rates and average balances
    for category, data in results.items():
        total = data['churned'] + data['retained']
        data['churn_rate'] = data['churned'] / total if total > 0 else 0
        data['avg_balance'] = data['total_balance'] / total if total > 0 else 0
    
    return results, enriched_customers

# Run the comprehensive analysis
print("🚀 Running Comprehensive Analysis Pipeline...\n")
churn_analysis, customer_list = comprehensive_analysis(df)

print("\n📊 Churn Analysis Results by Age Category:")
print("=" * 60)
for category, stats in churn_analysis.items():
    print(f"\n🏷️  {category}:")
    print(f"   Churned: {stats['churned']}")
    print(f"   Retained: {stats['retained']}")
    print(f"   Churn Rate: {stats['churn_rate']:.2%}")
    print(f"   Avg Balance: ${stats['avg_balance']:,.2f}")

print(f"\n✅ Total customers processed: {len(customer_list)}")

🚀 Running Comprehensive Analysis Pipeline...

🚀 Starting comprehensive_analysis...
✅ Finished comprehensive_analysis – processed 2 rows
⏱️  wrapper took 1.0638 seconds

📊 Churn Analysis Results by Age Category:

🏷️  Adult:
   Churned: 481
   Retained: 2028
   Churn Rate: 19.17%
   Avg Balance: $119,280.74

🏷️  Young:
   Churned: 34
   Retained: 483
   Churn Rate: 6.58%
   Avg Balance: $119,362.11

🏷️  Senior:
   Churned: 40
   Retained: 212
   Churn Rate: 15.87%
   Avg Balance: $118,447.30

✅ Total customers processed: 3278


## 🎉 What We Achieved Today

✅ **Day 1 Skills:** Used list comprehensions, generators, and dict comprehensions for efficient data filtering

✅ **Day 2 Skills:** Applied `map`, `filter`, `reduce`, and `partial` for functional data transformations

✅ **Day 3 Skills:** Created decorators for timing/logging and closures for reusable filter factories

✅ **Combined Pipeline:** Built a complete preprocessing workflow using ALL concepts together

---

## 💡 How These Python Tools Help

- **🚀 Faster:** Generators process data lazily without loading everything into memory
- **🧹 Cleaner:** Comprehensions and functional tools make code readable and concise
- **🔧 More Reusable:** Decorators and closures let you build modular, reusable components
- **📈 Professional:** These are the exact tools data scientists use for ML preprocessing!

---

## 📝 Mini Task for You

Create a **decorated function** that calculates the **average balance per geography** using `reduce`:

```python
@timing_decorator
@logging_decorator
def avg_balance_by_geography(dataframe):
    # Use reduce to group by Geography and calculate average Balance
    pass
```

**Hint:** Your accumulator should track sum and count per geography, then calculate averages at the end!

---
